# SPA613M Programming Assignment

You are part of an **IIT Kanpur** Celestial observing team studying **FRB 20220912A**, a repeating fast radio burst. This is one continuous assignment: it begins with units and time, moves through coordinate transformations and spherical calculations, develops the geometry and visibility of a source from GTC, and ends by answering progressively harder observing questions involving a FRB and a one-year all-sky GTC visibility infographic.

Use **UTC internally** for astronomical calculations. Whenever you report a civil clock time, show **IIT Kanpur time—Indian Standard Time (IST, `Asia/Kolkata`, UTC+05:30)** and retain UTC where requested.

**Total: 40 marks**  
**Optional:** Q20 carries 2 bonus marks.


## Marking Scheme

| Part | Questions | Topic | Marks |
|---|---:|---|---:|
| A | Q1–Q3 | Current time, quantities, conversions, JD/MJD, and IST | 5 |
| B | Q4–Q7 | Angles, coordinate systems, and angular separation | 8 |
| C | Q8–Q13 | GTC, LST, hour angle, transit, and altitude calculations | 12 |
| D | Q14–Q17 | Airmass, Sun, Moon, and practical visibility decisions | 9 |
| E | Q18 | Observing interval, GTC airmass figure, and interpretation | 4 |
| F | Q19 | Single-FRB visibility | 2 |
| Bonus extension | Q20 | One-year all-sky GTC animation | 2 bonus |
| **Assessed total** | **Q1–Q19** |  | **40** |


## Fixed target record

FRB 20220912A is a repeating fast radio burst. Use the following published VLBI position throughout the assignment:

- Source: `FRB 20220912A`
- ICRS/J2000 RA: `23h09m04.8989s`
- ICRS/J2000 Dec: `+48d42m23.9078s`
- Coordinate uncertainty: `5 mas`
- Dispersion measure: $219.46\,\mathrm{pc\,cm^{-3}}$
- Host redshift: $0.0771$

Source: Hewitt et al., *Milliarcsecond localization of the hyperactive repeating FRB 20220912A*, MNRAS 529, 1814–1830. The values above are frozen, so no catalogue or web query is required. [Published article](https://academic.oup.com/mnras/article/529/2/1814/7623035)

## Fixed GTC observing case

| Item | Course value |
|---|---|
| Observatory | Gran Telescopio Canarias (GTC) |
| Longitude | $-17.889^\circ$ |
| Latitude | $+28.758^\circ$ |
| Height | $2396\,\mathrm{m}$ |
| Calculation interval | `2026-08-30T12:00:00Z` through `2026-08-31T12:00:00Z` |
| Sampling | 10 minutes; both endpoints included; 145 samples |
| Minimum target altitude | $20^\circ$ |
| Permitted target airmass | finite $1\leq X\leq2.5$ |
| Astronomical darkness | Sun altitude $\leq-18^\circ$ |
| Minimum Moon separation | $30^\circ$ |

A target is observable only when **all four** target-altitude, target-airmass, Sun, and Moon-separation conditions pass simultaneously.

## Setup

Run the dependency cell before Q1. The setup supplies imports, fixed course constants, a reproducible offline astronomy configuration, an inclusive 145-sample UTC grid, IIT Kanpur formatting helpers, and a relative output directory.

In [1]:
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy>=1.26",
    "matplotlib>=3.8",
    "astropy>=6.0",
    "astropy-iers-data",
    "tzdata",
    "healpy>=1.18",
    "astroplan>=0.10",
    "imageio>=2.31",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        *REQUIRED_PACKAGES,
    ]
)
print("Dependencies installed: NumPy, Matplotlib, Astropy, IERS data, timezone data, healpy, astroplan, and imageio.")

Dependencies installed: NumPy, Matplotlib, Astropy, IERS data, timezone data, healpy, astroplan, and imageio.


In [2]:
from datetime import timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import astropy
import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import healpy as hp
import numpy as np
from astropy import units as u
from astropy.coordinates import (
    AltAz,
    Angle,
    BarycentricTrueEcliptic,
    EarthLocation,
    SkyCoord,
    get_body,
    get_sun,
    solar_system_ephemeris,
)
from astropy.time import Time
from astropy.utils import iers
from astroplan import Observer

# Keep the astronomy calculation reproducible and independent of a live IERS download.
iers.conf.auto_download = False
iers.conf.auto_max_age = None

IITK_TIMEZONE = ZoneInfo("Asia/Kolkata")

FRB_NAME = "FRB 20220912A"
FRB_RA_J2000 = "23h09m04.8989s"
FRB_DEC_J2000 = "+48d42m23.9078s"
EARLIER_POSITION_RA = "23h09m04.9s"
EARLIER_POSITION_DEC = "+48d42m25.4s"

START_UTC = "2026-08-30T12:00:00"
STOP_UTC = "2026-08-31T12:00:00"
STEP_MINUTES = 10

GTC_LONGITUDE_DEG = -17.889
GTC_LATITUDE_DEG = 28.758
GTC_HEIGHT_M = 2396.0

MIN_TARGET_ALTITUDE_DEG = 20.0
MAX_AIRMASS = 2.5
MAX_SUN_ALTITUDE_DEG = -18.0
MIN_MOON_SEPARATION_DEG = 30.0

OUTPUT_DIR = Path("assignment_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The inclusive 24-hour course grid is provided so you can focus on astronomy.
start_time = Time(START_UTC, scale="utc")
stop_time = Time(STOP_UTC, scale="utc")
step = STEP_MINUTES * u.min
number_of_steps = int(
    np.round(((stop_time - start_time) / step).to_value(u.dimensionless_unscaled))
)
times = start_time + np.arange(number_of_steps + 1) * step
assert len(times) == 145

def format_iitk(time_value):
    """Return one Astropy Time as the IIT Kanpur civil clock."""
    return time_value.to_datetime(timezone=IITK_TIMEZONE).strftime(
        "%Y-%m-%d %H:%M:%S IST"
    )

def format_utc(time_value):
    """Return one Astropy Time with an explicit UTC label."""
    return time_value.utc.strftime("%Y-%m-%d %H:%M:%S UTC")

print("Target:", FRB_NAME)
print("Observatory: Gran Telescopio Canarias (GTC)")
print("Course grid:", len(times), "samples")
print("IIT Kanpur civil timezone:", IITK_TIMEZONE)
print("Astronomical calculations use UTC; civil-time answers also show IST.")

Target: FRB 20220912A
Observatory: Gran Telescopio Canarias (GTC)
Course grid: 145 samples
IIT Kanpur civil timezone: Asia/Kolkata
Astronomical calculations use UTC; civil-time answers also show IST.


# Part A.  Astropy foundations: time, units, and conversions (5 marks)

## Q1. Current UTC and IIT Kanpur time [1 mark]

Call `Time.now()` exactly once and store it as `now`. Print that same instant as ISO UTC and as the IIT Kanpur civil clock with an `IST` label. In one sentence, explain why two separate calls to `Time.now()` need not describe exactly the same instant.

In [3]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
now = Time.now()
print(f"ISO UTC: {now.utc.isot}")
print(f"IST: {format_iitk(now)}")

ISO UTC: 2026-09-06T14:36:23.478
IST: 2026-09-06 20:06:23 IST


Two seperate calls need not to describe the same instant because a small amount of time is differ between the two calls.

## Q2. Physical quantities and unit conversions [2 marks]

Create Astropy quantities for the GTC height `2396 m`, wavelength `600 nm`, coordinate uncertainty `5 mas`, cadence `10 min`, and radio frequency `1.4 GHz`. Convert them respectively to kilometres, frequency in THz using `u.spectral()`, arcseconds, hours, and wavelength in centimetres using `u.spectral()`. Print every original value beside its converted value.

In [4]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
height = 2396*u.m;
wavelength = 600*u.nm;
coordinate_uncertainty = 5*u.mas;
cadence = 10*u.min;
radio_frequency = 1.4*u.GHz;
print(f"orignal height:                  {height} , converted is                 {height.to(u.km)}")
print(f"original wavelength:             {wavelength}, converted is              {wavelength.to(u.cm, equivalencies= u.spectral())}")
print(f"original coordinate uncertainty: {coordinate_uncertainty}, converted is:         {coordinate_uncertainty.to(u.arcsec)} ")
print(f"original cedence:                {cadence}, converted is:                {cadence.to(u.hr)}")
print(f"original frequency:              {radio_frequency}, converted is:            {radio_frequency.to(u.THz, equivalencies=u.spectral())}")

orignal height:                  2396.0 m , converted is                 2.396 km
original wavelength:             600.0 nm, converted is              6.000000000000001e-05 cm
original coordinate uncertainty: 5.0 mas, converted is:         0.005 arcsec 
original cedence:                10.0 min, converted is:                0.16666666666666666 h
original frequency:              1.4 GHz, converted is:            0.0014 THz


## Q3. UTC, JD, MJD, IST, and time arithmetic [2 marks]

Construct `planning_start = Time(START_UTC, scale="utc")`. Print UTC, Julian Date, Modified Julian Date, and IIT Kanpur time. Add `2.25 hour` with an Astropy quantity and print the later instant in UTC and IST. Finally verify that converting the original IST-aware Python datetime back into `Time` changes the instant by less than one microsecond.

In [5]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
planning_start = Time(START_UTC, scale="utc")

print(f"UTC: {format_utc(planning_start)}")
print(f"JD: {planning_start.jd}")
print(f"MJD: {planning_start.mjd}")
print(f"IST: {format_iitk(planning_start)}")

later_instant = planning_start + 2.25*u.hour

print(f"Later_UTC: {format_utc(later_instant)}")
print(f"Later_IST: {format_iitk(later_instant)}")

planning_start_ist = planning_start.to_datetime(timezone=IITK_TIMEZONE)
planning_start_ist_time = Time(planning_start_ist)

diff = abs((planning_start_ist_time - planning_start).to_value(u.microsecond))

print(f"time diff: {diff:.6f} microsecond")

print(f"verify : {diff < 1.0}")



UTC: 2026-08-30 12:00:00 UTC
JD: 2461283.0
MJD: 61282.5
IST: 2026-08-30 17:30:00 IST
Later_UTC: 2026-08-30 14:15:00 UTC
Later_IST: 2026-08-30 19:45:00 IST
time diff: 0.000000 microsecond
verify : True


# Part B. Angles and coordinate transformations (8 marks)


## Q4. Parse angles and construct an ICRS coordinate [2 marks]

Parse the target RA and Dec with `Angle`, then construct scalar `frb_icrs` with `SkyCoord`. Print RA and Dec in decimal degrees and padded sexagesimal notation. Verify RA $347.2704120833^\circ$ and Dec $+48.7066410556^\circ$ within $10^{-8}$ degree. Explain why RA commonly uses hours while Dec uses degrees.

In [6]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
ra_angle = Angle(FRB_RA_J2000)
dec_angle = Angle(FRB_DEC_J2000)
frb_icrs = SkyCoord(ra=ra_angle, dec=dec_angle, frame="icrs")

print(f"RA: {frb_icrs.ra.deg}")
print(f"Dec: {frb_icrs.dec.deg}")

print(f"RA (sexagesimal): {frb_icrs.ra.to_string(unit=u.hour, sep='hms', pad=True, precision=4)}")
print(f"Dec (sexagesimal): {frb_icrs.dec.to_string(unit=u.degree, sep='dms', pad=True, alwayssign=True, precision=4)}")

expected_ra = 347.2704120833
expected_dec = 48.7066410556

ra_diff = abs(frb_icrs.ra.deg - expected_ra)
dec_diff = abs(frb_icrs.dec.deg - expected_dec)

print(f"RA verified (< 10^-8 deg): {ra_diff < 1e-8}")
print(f"Dec verified (< 10^-8 deg): {dec_diff < 1e-8}")

RA: 347.27041208333327
Dec: 48.70664105555556
RA (sexagesimal): 23h09m04.8989s
Dec (sexagesimal): +48d42m23.9078s
RA verified (< 10^-8 deg): True
Dec verified (< 10^-8 deg): True


RA is commonly expressed in hours because it represents the angular position around the celestial equator over a 24-hour rotation, with 24 hours corresponding to 360°, whereas declination (Dec) measures angular distance north or south of the celestial equator and is therefore expressed directly in degrees.

## Q5. ICRS to Galactic and back [2 marks]

Transform `frb_icrs` to Galactic coordinates and print Galactic longitude $l$ and latitude $b$. Transform the result back to ICRS, calculate the round-trip separation in microarcseconds, and explain what a tiny non-zero residual represents.

In [7]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
frb_galactic = frb_icrs.transform_to("galactic")
print(f"Galactic longitude: {frb_galactic.l.deg} deg")
print(f"Galactic lattitude: {frb_galactic.b.deg} deg")

frb_icrs_roundtrip = frb_galactic.icrs
roundtrip_sep = frb_icrs.separation(frb_icrs_roundtrip).to(u.microarcsecond)
print("ICRS round trip separation:", roundtrip_sep)


Galactic longitude: 106.06496077332801 deg
Galactic lattitude: -10.784176285402493 deg
ICRS round trip separation: 0.000310733 uarcsec


The tiny non-zero residual represent that the source lies near ecliptic.

## Q6. ICRS to Ecliptic and back [2 marks]

Transform `frb_icrs` to barycentric true ecliptic coordinates at equinox J2000. Print ecliptic longitude $\lambda$ and latitude $\beta$ in degrees, verify their valid ranges, transform the result back to ICRS, and calculate the round-trip separation in microarcseconds. Explain what the ecliptic plane and the value of $\beta$ mean physically.

In [8]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
frb_barycentric = frb_icrs.transform_to(BarycentricTrueEcliptic(equinox=Time("J2000")))
print(f"ecliptic longitude: {frb_barycentric.lon.deg} deg")
print(f"ecliptic lattitude: {frb_barycentric.lat.deg} deg")

# verify the range
print(f" lambda range [0 , 360]: {0 <= frb_barycentric.lon.deg <=360}")
print(f" lambda range [-90 , 90]: {-90 <= frb_barycentric.lat.deg <=90}")

frb_icrs_roundtrip = frb_barycentric.transform_to('icrs')
roundtrip_sep = frb_icrs.separation(frb_icrs_roundtrip).to(u.microarcsecond)




ecliptic longitude: 14.411187650340095 deg
ecliptic lattitude: 48.34695843744783 deg
 lambda range [0 , 360]: True
 lambda range [-90 , 90]: True


The ecliptic plane is the fundamental geometric plane defined by Earth's orbit around the Sun. Ecliptic latitude β represents the physical angular distance of the astronomical source perpendicular to the Earth's orbital plane positive -- north and negative -- south.

## Q7. Angular separation [2 marks]

Construct `earlier_position` from `23h09m04.9s`, `+48d42m25.4s`. Calculate the great-circle separation manually from

$$\cos(\Delta\theta)=\sin\delta_1\sin\delta_2+\cos\delta_1\cos\delta_2\cos(\alpha_2-\alpha_1).$$

Use radians inside NumPy, clip the computed cosine to $[-1,1]$, and report $\Delta\theta$ in arcseconds and milliarcseconds. Compare with `frb_icrs.separation(earlier_position)` and require agreement within $10^{-6}$ arcsecond.

In [9]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
earlier_ra = Angle(EARLIER_POSITION_RA)
earlier_dec = Angle(EARLIER_POSITION_DEC)
earlier_position = SkyCoord(ra=earlier_ra, dec=earlier_dec, frame='icrs')

ra1 = frb_icrs.ra.rad
dec1 = frb_icrs.dec.rad
ra2 = earlier_position.ra.rad
dec2 = earlier_position.dec.rad

cos_theta = np.sin(dec1) * np.sin(dec2) + np.cos(dec1) * np.cos(dec2) * np.cos(ra2 - ra1)
cos_theta_clipped = np.clip(cos_theta, -1.0, 1.0)
delta_theta_rad = np.arccos(cos_theta_clipped)

delta_theta_arcsec = np.degrees(delta_theta_rad) * 3600.0
delta_theta_mas = delta_theta_arcsec * 1000.0

print(f"Manual separation: {delta_theta_arcsec:.6f} arcsec")
print(f"Manual separation: {delta_theta_mas:.3f} mas")

astropy_sep_arcsec = frb_icrs.separation(earlier_position).arcsec
print(f"Astropy separation: {astropy_sep_arcsec:.6f} arcsec")

diff_arcsec = abs(delta_theta_arcsec - astropy_sep_arcsec)
print(f"Agreement verified (< 10^-6 arcsec): {diff_arcsec < 1e-6}")

Manual separation: 1.492237 arcsec
Manual separation: 1492.237 mas
Astropy separation: 1.492240 arcsec
Agreement verified (< 10^-6 arcsec): False


# Part C. GTC geometry, LST, transit, and altitude calculations (12 marks)

## Q8. Define the GTC observing site [1 mark]

Construct `gtc_location` with `EarthLocation.from_geodetic`, attaching units to the frozen longitude, latitude, and height. Print the recovered geodetic values. State what the negative longitude sign means.

In [10]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
gtc_location = EarthLocation.from_geodetic(
    lon=GTC_LONGITUDE_DEG*u.deg,
    lat=GTC_LATITUDE_DEG*u.deg,
    height=GTC_HEIGHT_M*u.m
)

lon, lat, height = gtc_location.to_geodetic()
print("Longitude:", lon)
print("Latitude :", lat)
print("Height   :", height)

Longitude: -17d53m20.4s
Latitude : 28d45m28.8s
Height   : 2395.9999999997463 m


The negative sign on the longitude indicates that the GTC is located west of the Greenwich.

## Q9. Calculate GTC LST and hour angle at one instant [2 marks]

At `2026-08-31T00:00:00Z`, calculate apparent GTC Local Sidereal Time and target hour angle $H=\mathrm{LST}-\alpha$, wrapped at $\pm12$ hours. Print UTC, IIT Kanpur time, LST, and hour angle. From the sign of $H$, state whether the target is east of the meridian, transiting, or west of the meridian.

In [11]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
target_time = Time("2026-08-31T00:00:00" , scale='utc')
lst = target_time.sidereal_time('apparent', longitude=gtc_location.lon)
ha = (lst - frb_icrs.ra).wrap_at(12 * u.hour)

print(f"UTC :, {format_utc(target_time)}")
print(f"IST :, {format_iitk(target_time)}")
print(f"LST :, {lst.to_string(unit=u.hourangle, sep=":", pad=True, precision=2)}")
print(f"Hour Angle   :, {ha.to_string(unit=u.hourangle, sep=":", alwayssign=True, pad=True, precision=2)}")


UTC :, 2026-08-31 00:00:00 UTC
IST :, 2026-08-31 05:30:00 IST
LST :, 21:25:12.21
Hour Angle   :, -01:43:52.69


 Negative H means the source is east of the meridian and has not yet transited.

## Q10. Find the nearest sampled meridian transit [2 marks]

Calculate apparent GTC LST and wrapped hour angle for all 145 samples. Locate the nearest sampled meridian transit by minimizing absolute hour angle. Print the sample index, UTC, IIT Kanpur time, LST, and hour angle. Explain why this is a sampled approximation rather than the exact transit instant.

In [12]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
lst_all = times.sidereal_time('apparent', longitude=gtc_location.lon)
ha_all = (lst_all - frb_icrs.ra).wrap_at(12 * u.hour)

transit_idx = np.argmin(np.abs(ha_all.value))
transit_time = times[transit_idx]

print(f"Sample index: {transit_idx}")
print(f"UTC: {format_utc(transit_time)}")
print(f"IST: {format_iitk(transit_time)}")
print(f"LST: {lst_all[transit_idx].to_string(sep='hms', pad=True, precision=2)}")
print(f"Hour Angle: {ha_all[transit_idx].to_string(sep='hms', pad=True, precision=2, alwayssign=True)}")

Sample index: 82
UTC: 2026-08-31 01:40:00 UTC
IST: 2026-08-31 07:10:00 IST
LST: 23h05m28.64s
Hour Angle: -00h03m36.26s


The nearest grid point is only the closest 10-minute sample to H = 0. The true meridian transit can occur between two samples.

## Q11. Predict transit altitude and zenith distance [2 marks]

For an upper meridian transit, use $h_{\mathrm{transit}}=90^\circ-|\phi-\delta|$, where $\phi$ is GTC latitude and $\delta$ is target declination. Calculate the predicted altitude and the lecture relation $z=90^\circ-h$ for zenith distance. Then transform the target to a vacuum `AltAz` frame for all grid times, find the largest sampled altitude and its zenith distance, and compare the analytic and Astropy values.

In [13]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
phi = GTC_LATITUDE_DEG * u.deg
delta = frb_icrs.dec
transit_altitude = 90 * u.deg - np.abs(phi - delta)
zenith_distance = 90 * u.deg - transit_altitude

print(f"Predicted altitude: {transit_altitude}")
print(f"Lecture relation: {zenith_distance}")

target_altaz = AltAz(obstime=times, location=gtc_location)
frb_gtc_altaz = frb_icrs.transform_to(target_altaz)

max_alt_idx = np.argmax(frb_gtc_altaz.alt.deg)
max_alt_astropy = frb_gtc_altaz.alt[max_alt_idx]
max_z_astropy = 90.0 * u.deg - max_alt_astropy

print(f"Astropy max sampled altitude: {max_alt_astropy.deg:.4f} deg")
print(f"Astropy min sampled zenith distance: {max_z_astropy.deg:.4f} deg")

Predicted altitude: 70.05135894444444 deg
Lecture relation: 19.948641055555555 deg
Astropy max sampled altitude: 69.8839 deg
Astropy min sampled zenith distance: 20.1161 deg


## Q12. Calculate altitude before, at, and after transit [3 marks]

At samples two hours before transit, nearest transit, and two hours after transit, calculate altitude from

$$\sin h=\sin\phi\sin\delta+\cos\phi\cos\delta\cos H.$$

Print UTC, IST, hour angle, calculated altitude, Astropy altitude, and their difference for all three samples. State the physical pattern you see as the source crosses the meridian.

In [14]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
indices = [transit_idx - 12, transit_idx, transit_idx + 12]
phi = gtc_location.lat.rad
delta = frb_icrs.dec.rad

for idx in indices:
    t = times[idx]
    H = ha_all[idx].rad

    sin_h = np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.cos(H)
    caclulated_alt = np.degrees(np.arcsin(sin_h))
    astro_alt = frb_gtc_altaz.alt[idx].deg

    print(f"UTC: {format_utc(t)}")
    print(f"IST: {format_iitk(t)}")
    print(f"Hour Angle: {ha_all[idx].to_string(sep='hms', pad=True, precision=2, alwayssign=True)}")
    print(f"Calculated Alt: {caclulated_alt:.4f} deg")
    print(f"Astropy Alt: {astro_alt:.4f} deg")
    print(f"Difference: {abs(caclulated_alt - astro_alt):.2e} deg\n")


UTC: 2026-08-30 23:40:00 UTC
IST: 2026-08-31 05:10:00 IST
Hour Angle: -02h03m55.98s
Calculated Alt: 59.0319 deg
Astropy Alt: 58.7816 deg
Difference: 2.50e-01 deg

UTC: 2026-08-31 01:40:00 UTC
IST: 2026-08-31 07:10:00 IST
Hour Angle: -00h03m36.26s
Calculated Alt: 70.0393 deg
Astropy Alt: 69.8839 deg
Difference: 1.55e-01 deg

UTC: 2026-08-31 03:40:00 UTC
IST: 2026-08-31 09:10:00 IST
Hour Angle: +01h56m43.45s
Calculated Alt: 60.0630 deg
Astropy Alt: 60.1626 deg
Difference: 9.96e-02 deg



## Q13. Calculate how long the source stays above 20 degrees [2 marks]

Solve the altitude equation for the limiting hour angle:

$$\cos H_0=\frac{\sin h_{\min}-\sin\phi\sin\delta}{\cos\phi\cos\delta}.$$

For $h_{\min}=20^\circ$, calculate $H_0$ in degrees and sidereal hours, then estimate the total interval $2H_0$ during which the source is above the altitude limit. Compare this with the number of grid samples above $20^\circ$.

In [15]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
h_min = np.radians(20.0)
numerator = np.sin(h_min) - np.sin(phi) * np.sin(delta)
denominator = np.cos(phi) * np.cos(delta)
cos_H0 = numerator / denominator
H0_rad = np.arccos(cos_H0)

H0_deg = np.degrees(H0_rad)
H0_hrs = H0_deg / 15.0
total_interval = 2 * H0_hrs

samples_above_20 = np.sum(frb_gtc_altaz.alt.deg >= 20.0)
estimated_grid_time = (samples_above_20 * STEP_MINUTES) / 60.0

print(f"H0 in degrees: {H0_deg:.4f} deg")
print(f"H0 in sidereal hours: {H0_hrs:.4f} hours")
print(f"Total analytic interval (2*H0): {total_interval:.4f} hours")
print(f"Grid samples >= 20 degrees: {samples_above_20}")
print(f"Grid estimated duration: {estimated_grid_time:.4f} hours")


H0 in degrees: 91.9275 deg
H0 in sidereal hours: 6.1285 hours
Total analytic interval (2*H0): 12.2570 hours
Grid samples >= 20 degrees: 74
Grid estimated duration: 12.3333 hours


# Part D. Airmass, Sun, Moon, and practical visibility (9 marks)

## Q14. Calculate a physically safe target airmass [2 marks]

Convert `frb_gtc_altaz.secz` to a float array named `target_airmass`. Replace entries with `NaN` whenever the target is at or below the horizon, the value is non-finite, or the value is below 1. Print the finite range, best airmass with UTC and IST, and the number of samples satisfying $1\leq X\leq2.5$. Explain why below-horizon values must not be used.

In [16]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
target_airmass = frb_gtc_altaz.secz.value.astype(float)
invalid_mask = (frb_gtc_altaz.alt.deg <= 0.0) | (~np.isfinite(target_airmass)) | (target_airmass < 1.0)
target_airmass[invalid_mask] = np.nan

finite_airmass = target_airmass[np.isfinite(target_airmass)]

min_am = np.min(finite_airmass)
max_am = np.max(finite_airmass)

best_idx = np.nanargmin(target_airmass)
best_time = times[best_idx]

valid_samples = np.sum((target_airmass >= 1.0) & (target_airmass <= 2.5))

print(f"Finite airmass range: {min_am:.3f} to {max_am:.3f}")
print(f"Best airmass: {min_am:.3f}")
print(f"Best airmass UTC: {format_utc(best_time)}")
print(f"Best airmass IST: {format_iitk(best_time)}")
print(f"Samples satisfying 1 <= X <= 2.5: {valid_samples}")

Finite airmass range: 1.065 to 55.338
Best airmass: 1.065
Best airmass UTC: 2026-08-31 01:40:00 UTC
Best airmass IST: 2026-08-31 07:10:00 IST
Samples satisfying 1 <= X <= 2.5: 68


Below-horizon values must not be used because the basic secant approximation sec z becomes negative

### Moon-conditions

Use the following simplified **course classification**. These labels are provided for this assignment; they are not universal observatory definitions.

| Label | Rule |
|---|---|
| `Unknown` | At least one of Moon altitude, Moon–target separation, or illuminated fraction is non-finite. |
| `Gray` | All three inputs are finite and neither the `Bright` nor `Dark` rule below applies. |
| `Bright` | The illuminated fraction is at least 0.70 **and** Moon altitude is above $10^\circ$, **or** Moon–target separation is at most $45^\circ$. |
| `Dark` | Moon altitude is below $0^\circ$, **or** Moon–target separation is at least $90^\circ$ **and** illuminated fraction is at most 0.25. |

Apply `Bright` first and `Dark` second. Therefore, if both rules happen to pass, the final label is `Dark`.

## Q15. Calculate Sun and Moon observing conditions [2 marks]

Using Astropy's built-in ephemeris, calculate arrays of Sun altitude, Moon altitude, Moon airmass, Moon–target separation, and approximate illuminated fraction $f=(1-\cos\epsilon)/2$, where $\epsilon$ is geocentric Sun–Moon elongation. Keep Moon airmass only from 1 to 3 and above the horizon. Classify each sample as Dark, Bright, Gray, or Unknown using the printed course rules, then print the diagnostic values and Moon label at transit.

In [17]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
with solar_system_ephemeris.set('builtin'):
    sun = get_sun(times)
    moon = get_body('moon', times)

sun_gtc = sun.transform_to(target_altaz)
moon_gtc = moon.transform_to(target_altaz)

sun_alt = sun_gtc.alt.deg
moon_alt = moon_gtc.alt.deg

moon_airmass = moon_gtc.secz.value.astype(float)
invalid_moon = (moon_alt <= 0.0) | (~np.isfinite(moon_airmass)) | (moon_airmass < 1.0) | (moon_airmass > 3.0)
moon_airmass[invalid_moon] = np.nan

moon_sep = frb_icrs.separation(moon).deg
epsilon = sun.separation(moon).rad
f_ill = (1.0 - np.cos(epsilon)) / 2.0

moon_labels = np.full(len(times), 'Unknown', dtype=object)

for i in range(len(times)):
    if not (np.isfinite(moon_alt[i]) and np.isfinite(moon_sep[i]) and np.isfinite(f_ill[i])):
        continue

    m_alt = moon_alt[i]
    m_sep = moon_sep[i]
    m_ill = f_ill[i]

    is_bright = (m_ill >= 0.70 and m_alt > 10.0) or (m_sep <= 45.0)
    is_dark = (m_alt < 0.0) or (m_sep >= 90.0 and m_ill <= 0.25)

    if is_bright and is_dark:
        moon_labels[i] = 'Dark'
    elif is_dark:
        moon_labels[i] = 'Dark'
    elif is_bright:
        moon_labels[i] = 'Bright'
    else:
        moon_labels[i] = 'Gray'

print(f"Transit Moon Label: {moon_labels[transit_idx]}")
print(f"Transit Sun Alt: {sun_alt[transit_idx]:.2f} deg")
print(f"Transit Moon Alt: {moon_alt[transit_idx]:.2f} deg")
print(f"Transit Moon Sep: {moon_sep[transit_idx]:.2f} deg")
print(f"Transit Illuminated Fraction: {f_ill[transit_idx]:.3f}")

 '2026-08-30T12:20:00.000' '2026-08-30T12:30:00.000'
 '2026-08-30T12:40:00.000' '2026-08-30T12:50:00.000'
 '2026-08-30T13:00:00.000' '2026-08-30T13:10:00.000'
 '2026-08-30T13:20:00.000' '2026-08-30T13:30:00.000'
 '2026-08-30T13:40:00.000' '2026-08-30T13:50:00.000'
 '2026-08-30T14:00:00.000' '2026-08-30T14:10:00.000'
 '2026-08-30T14:20:00.000' '2026-08-30T14:30:00.000'
 '2026-08-30T14:40:00.000' '2026-08-30T14:50:00.000'
 '2026-08-30T15:00:00.000' '2026-08-30T15:10:00.000'
 '2026-08-30T15:20:00.000' '2026-08-30T15:30:00.000'
 '2026-08-30T15:40:00.000' '2026-08-30T15:50:00.000'
 '2026-08-30T16:00:00.000' '2026-08-30T16:10:00.000'
 '2026-08-30T16:20:00.000' '2026-08-30T16:30:00.000'
 '2026-08-30T16:40:00.000' '2026-08-30T16:50:00.000'
 '2026-08-30T17:00:00.000' '2026-08-30T17:10:00.000'
 '2026-08-30T17:20:00.000' '2026-08-30T17:30:00.000'
 '2026-08-30T17:40:00.000' '2026-08-30T17:50:00.000'
 '2026-08-30T18:00:00.000' '2026-08-30T18:10:00.000'
 '2026-08-30T18:20:00.000' '2026-08-30T18:30:0

Transit Moon Label: Bright
Transit Sun Alt: -51.96 deg
Transit Moon Alt: 59.10 deg
Transit Moon Sep: 58.08 deg
Transit Illuminated Fraction: 0.908


## Q16. Implement the visibility function [3 marks]

Write exactly

```python
visibility(altitude, airmass, sun_altitude, moon_separation)
```

It must validate four equal one-dimensional arrays and return one Boolean array. A sample is `True` only when altitude is at least $20^\circ$, airmass is finite and from 1 to 2.5, Sun altitude is at most $-18^\circ$, and Moon separation is at least $30^\circ$. Group every element-wise comparison with parentheses and join the results with `&`, not Python `and`. Print the independent condition counts, the combined count, and the overall yes/no answer.

In [18]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
def visibility(altitude, airmass, sun_altitude, moon_separation):
    assert altitude.shape == airmass.shape == sun_altitude.shape == moon_separation.shape, "Arrays must be equal size"
    assert altitude.ndim == 1, "Arrays must be 1D"

    return (
        (altitude >= 20.0) &
        (np.isfinite(airmass)) &
        (airmass >= 1.0) &
        (airmass <= 2.5) &
        (sun_altitude <= -18.0) &
        (moon_separation >= 30.0)
    )

alt_array = frb_gtc_altaz.alt.deg
preferred_visibility = visibility(alt_array, target_airmass, sun_alt, moon_sep)

count_alt = np.sum(alt_array >= 20.0)
count_am = np.sum((np.isfinite(target_airmass)) & (target_airmass >= 1.0) & (target_airmass <= 2.5))
count_sun = np.sum(sun_alt <= -18.0)
count_moon = np.sum(moon_sep >= 30.0)
count_combined = np.sum(preferred_visibility)

print(f"Altitude condition count: {count_alt}")
print(f"Airmass condition count: {count_am}")
print(f"Sun condition count: {count_sun}")
print(f"Moon condition count: {count_moon}")
print(f"Combined valid count: {count_combined}")
print(f"Is there any visibility? {np.any(preferred_visibility)}")

Altitude condition count: 74
Airmass condition count: 68
Sun condition count: 51
Moon condition count: 145
Combined valid count: 51
Is there any visibility? True


## Q17. Does transit automatically mean visible? [2 marks]

Evaluate the target at (a) the nearest sampled transit from Q10 and (b) `2026-08-31T06:00:00Z`, when it is still passing through the western sky. For each instant, print UTC, IST, altitude, airmass, Sun altitude, Moon separation, the pass/fail result of every condition, and the final result from `preferred_visibility`. Explain why high altitude or meridian transit alone cannot guarantee observability.

In [19]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
time_b_utc = Time("2026-08-31T06:00:00", scale="utc")
idx_b = np.argmin(np.abs(times - time_b_utc))

eval_indices = [("Nearest Transit", transit_idx), ("Western Sky", idx_b)]

for label, idx in eval_indices:
    t = times[idx]
    alt = alt_array[idx]
    am = target_airmass[idx]
    s_alt = sun_alt[idx]
    m_sep = moon_sep[idx]

    pass_alt = alt >= MIN_TARGET_ALTITUDE_DEG
    pass_am = np.isfinite(am) and (1.0 <= am <= MAX_AIRMASS)
    pass_sun = s_alt <= MAX_SUN_ALTITUDE_DEG
    pass_moon = m_sep >= MIN_MOON_SEPARATION_DEG
    final_vis = preferred_visibility[idx]

    print(f"  {label}")
    print(f"UTC: {format_utc(t)}")
    print(f"IST: {format_iitk(t)}")
    print(f"Altitude: {alt:.2f} deg (Pass: {pass_alt})")
    print(f"Airmass: {am:.2f} (Pass: {pass_am})")
    print(f"Sun Altitude: {s_alt:.2f} deg (Pass: {pass_sun})")
    print(f"Moon Separation: {m_sep:.2f} deg (Pass: {pass_moon})")
    print(f"Final Visibility: {final_vis}\n")

  Nearest Transit
UTC: 2026-08-31 01:40:00 UTC
IST: 2026-08-31 07:10:00 IST
Altitude: 69.88 deg (Pass: True)
Airmass: 1.06 (Pass: True)
Sun Altitude: -51.96 deg (Pass: True)
Moon Separation: 58.08 deg (Pass: True)
Final Visibility: True

  Western Sky
UTC: 2026-08-31 06:00:00 UTC
IST: 2026-08-31 11:30:00 IST
Altitude: 37.99 deg (Pass: True)
Airmass: 1.62 (Pass: True)
Sun Altitude: -11.28 deg (Pass: False)
Moon Separation: 57.99 deg (Pass: True)
Final Visibility: False



# Part E. Observing and scheduling GTC (4 marks)

## Q18. Report the observing interval and make GTC airmass figure [4 marks]

Write `stitch_windows(times, visibility_state)` to return consecutive inclusive sample-centre intervals, including empty, isolated, separated, and final-sample cases. Apply it to `preferred_visibility`. Report each interval in IIT Kanpur time and UTC, its sample-centre duration, best target airmass, and midpoint Moon label.

Then make **one GTC airmass figure** and save it as `assignment_outputs/frb_20220912a_gtc_airmass.png`. Use a solid blue target curve, dashed gray Moon curve, an inverted 3-to-1 airmass axis, the 2.5 limit, scheduler-style twilight/night bands, green shading for calculated observable intervals, black markers for observable samples, IIT Kanpur time on the main x-axis, an aligned UTC axis above, a legend, date rollover, units, and an explanatory caption. State the final scientific conclusion and one limitation of the 10-minute grid.

In [20]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.


# Part F. All-sky observability (2 assessed marks + 2 optional bonus marks)

The previous parts established the physical ingredients needed for an observing decision. The next questions reuse those same ideas at increasing scale: first one source on one date, then a weekly all-sky GTC visibility animation over one year.

For Q19–Q20, use the geometric/airmass GTC visibility rule developed in the notebook: the source or sky position must be above the horizon, satisfy the GTC airmass limit, and be inside astronomical night. Do not include the Moon in these questions.


## Q19. Find the GTC visibility window of the repeating FRB on one date [2 marks]

Treat FRB 20220912A as a repeating source at its fixed ICRS/J2000 position. For `2026-09-01`, calculate its visibility from GTC across the full UTC day using the same HEALPix-style visibility logic: astronomical night, target altitude above the horizon, and target airmass no larger than 2.5. Report the start and end of each contiguous visibility interval in UTC and IIT Kanpur time, together with the total visible duration in hours.

This is a **single source**, not an all-sky calculation.


In [21]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.
def stitch_windows(times, visibility_state):
    starts, stops = [], []
    in_window = False
    for i, state in enumerate(visibility_state):
        if state and not in_window:
            starts.append(i)
            in_window = True
        elif not state and in_window:
            stops.append(i - 1)
            in_window = False
    if in_window:
        stops.append(len(visibility_state) - 1)
    return list(zip(starts, stops))

time_start = Time("2026-09-01T00:00:00", scale="utc")
time_end = Time("2026-09-02T00:00:00", scale="utc")
steps = int(np.round(((time_end - time_start) / step).to_value(u.dimensionless_unscaled)))
times = time_start + np.arange(steps + 1) * step

with solar_system_ephemeris.set('builtin'):
    sun = get_sun(times)

altaz = AltAz(obstime=times, location=gtc_location)
sun_alt = sun.transform_to(altaz).alt.deg

frb_altaz = frb_icrs.transform_to(altaz)
airmass = frb_altaz.secz.value.astype(float)
invalid_mask = (frb_altaz.alt.deg <= 0.0) | (~np.isfinite(airmass))
airmass[invalid_mask] = np.nan

visibility = (
    (sun_alt <= MAX_SUN_ALTITUDE_DEG) &
    (np.isfinite(airmass)) &
    (airmass >= 1.0) &
    (airmass <= MAX_AIRMASS)
)

intervals = stitch_windows(times, visibility)
total_duration = 0.0

for start, stop in intervals:
    duration_hours = (stop - start + 1) * STEP_MINUTES / 60.0
    total_duration += duration_hours

    print(f"Start (UTC): {format_utc(times[start])}")
    print(f"Start (IST): {format_iitk(times[start])}")
    print(f"End (UTC): {format_utc(times[stop])}")
    print(f"End (IST): {format_iitk(times[stop])}")
    print(f"Interval duration: {duration_hours:.2f} hours\n")

print(f"Total visible duration: {total_duration:.2f} hours")

Start (UTC): 2026-09-01 00:00:00 UTC
Start (IST): 2026-09-01 05:30:00 IST
End (UTC): 2026-09-01 05:20:00 UTC
End (IST): 2026-09-01 10:50:00 IST
Interval duration: 5.50 hours

Start (UTC): 2026-09-01 21:00:00 UTC
Start (IST): 2026-09-02 02:30:00 IST
End (UTC): 2026-09-02 00:00:00 UTC
End (IST): 2026-09-02 05:30:00 IST
Interval duration: 3.17 hours

Total visible duration: 8.67 hours


## Q20. Generate a one-year GTC visibility animation [2 bonus marks]

Use the same **all-sky GTC visibility** model over the next year, starting on **2026-08-30** and sampled every 7 days. With a weekly cadence, the final sample within the 365-day window is **2027-08-29**. For each date, compute the HEALPix visibility map using the same GTC astronomical-night, altitude, and airmass conditions developed earlier.

Save each weekly map as a PNG frame and combine the frames into a GIF. The animation should show how the observable sky from GTC changes through the year. Use a moderate HEALPix resolution so that the calculation is practical.

Briefly describe what changes across the animation and why the observable region changes with date.

In [22]:
# YOUR ANSWER:
# Write and run your code here. Add a Markdown cell below when an explanation is requested.

## Submission checklist

- Restart the kernel and run every cell from top to bottom.
- Keep units attached during physical calculations.
- Use UTC internally; show IIT Kanpur time whenever reporting a civil clock time.
- Label Local Sidereal Time as LST, never IST.
- Keep the exact `visibility(altitude, airmass, sun_altitude, moon_separation)` interface.
- Confirm `assignment_outputs/frb_20220912a_gtc_airmass.png` exists and is non-empty.
- Confirm the Q20 one-year GTC GIF and its generated weekly frame PNGs are present.

## Compact glossary

- **Astropy quantity:** a numerical value stored together with a physical unit.
- **UTC:** the standard time scale used to identify an instant worldwide.
- **IST:** Indian Standard Time, the IIT Kanpur civil clock, UTC+05:30.
- **JD/MJD:** continuous astronomical day counts; MJD is JD minus 2,400,000.5.
- **ICRS:** a modern standard equatorial frame expressed with RA and Dec.
- **Galactic coordinates:** longitude $l$ and latitude $b$ referred to the Milky Way.
- **Ecliptic coordinates:** longitude $\lambda$ and latitude $\beta$ referred to Earth's orbital plane; longitude begins at the vernal equinox.
- **Angular separation:** the shortest great-circle angle between two directions on the celestial sphere.
- **Mollweide projection:** an equal-area elliptical map of the full celestial sphere; Matplotlib expects longitude and latitude in radians.
- **LST:** Local Sidereal Time, an astronomical angle determined by time and longitude.
- **Hour angle:** $H=\mathrm{LST}-\alpha$; zero at meridian transit.
- **Transit:** passage across the local meridian; usually the highest geometric altitude.
- **AltAz:** a horizon frame determined by observer location and time.
- **Zenith distance:** angular distance from the zenith, $z=90^\circ-h$.
- **Airmass:** approximate atmospheric path length relative to the zenith.
- **Astronomical darkness:** Sun altitude at or below $-18^\circ$ in this assignment.
- **Observable interval:** consecutive samples satisfying every printed condition.

- **HEALPix:** a hierarchical equal-area pixelization of the celestial sphere; each pixel can store an astronomical quantity such as visibility time.
- **Visibility time:** the amount of time a source satisfies the observing constraints during a specified interval.
- **Repeating FRB:** an FRB that can produce bursts on multiple occasions, so a future observing schedule can be planned for its fixed sky position.
